# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 3 |
| **Integrantes** | Alvaro Enrique Muñoz Giraldo |
| **Caso de estudio** | Wanderbricks |
| **Fecha de entrega** | domingo 20 de septiembre |
| **🎥 Enlace al video** | [Clic aquí para ver el video](https://drive.google.com/file/d/1FKjXsciljcu9vYXZVd9olNhR58aji_e-/view?usp=sharing) |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*Qué necesidad de infraestructura plantea el caso y qué debe soportar el entorno.*
El proyecto Wanderbricks requiere una infraestructura analítica centralizada en la nube que soporte la ingesta, transformación y consumo de datos. El reto principal es gobernar este entorno mediante arquitectura Medallion, garantizando que cada rol acceda solo a las capas permitidas y mitigando la carga operativa de administrar servidores físicos.

---
## 2. Descripción de los datos

*Qué va a vivir en esta infraestructura: volumen esperado, frecuencia de actualización
y quién la va a consumir.*
La infraestructura procesará información estructurada y archivos CSV generados por las operaciones de Wanderbricks. El volumen tendrá una actualización diaria automatizada. Los datos limpios serán consumidos por herramientas de BI y analistas de negocio mediante consultas SQL.

## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura


[ Fuentes CSV/APIs ] → [ Ingesta PySpark ] → [ Capas Bronze/Silver/Gold ] → [ Procesamiento Jobs ] → [ Consumo BI ]
Administrado por el proveedor (Databricks/Nube): Cómputo elástico (clusters), almacenamiento de objetos subyacente (S3/Blob), alta disponibilidad y seguridad física.

Administrado por el equipo: Gobierno (Unity Catalog, esquemas), permisos (GRANT), modelado en capas (Medallion) y lógica de los pipelines.

3.2 Matriz de roles
Rol	Bronce	Plata	Oro
Analista	Sin acceso	Solo Lectura (SELECT)	Lectura Completa (SELECT / BI)
Ingeniero de datos	Control Total (ALL)	Control Total (ALL)	Lectura / Creación de vistas
Administrador	Control Total (ALL)	Control Total (ALL)	Control Total (ALL)
3.3 Especificación del equivalente IaaS
Máquinas y dimensionamiento: 3 instancias virtuales (ej. EC2 t3.xlarge) con 4 vCPUs y 16 GB RAM (1 Nodo Master, 2 Nodos Workers).

Sistema operativo: Ubuntu Server 22.04 LTS.

Software: Apache Spark standalone, Java JDK 11, Python 3.10+, conectores Delta.

Red y almacenamiento: Configuración manual de VPC, subredes, firewalls (Security Groups), discos de red EBS montados y sistema de archivos HDFS.

Esfuerzo operativo: Muy alto. Requiere aprovisionamiento manual, aplicación de parches de seguridad, monitoreo y carece de auto-escalado nativo.

3.4 Comparación IaaS / PaaS / SaaS
Criterio	IaaS	PaaS (Databricks)	SaaS
Control	Total sobre SO y redes	Alto sobre datos, nulo en hardware	Mínimo
Tiempo hasta el primer resultado	Lento (días de configuración)	Rápido (minutos)	Inmediato
Esfuerzo operativo	Muy alto (mantenimiento)	Bajo (gestionado)	Nulo
Costo	Fijo por servidor encendido	Pago por uso elástico	Suscripción fija
Escalabilidad	Manual y compleja	Automática (Auto-scaling)	Gestionada por el proveedor
Gobierno	Manual (scripts/ACLs)	Centralizado (Unity Catalog)	Integrado pero rígido
Conclusión: Para este caso de estudio analítico se selecciona PaaS (Databricks) porque elimina la carga técnica de administrar clústeres distribuidos en IaaS, permitiendo enfocar el esfuerzo de ingeniería en el valor de negocio, la calidad de datos y el gobierno centralizado.

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# Creación de catálogo y esquemas organizados (Arquitectura Medallion)
spark.sql("CREATE CATALOG IF NOT EXISTS wanderbricks_catalog;")
spark.sql("USE CATALOG wanderbricks_catalog;")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Capa de datos crudos ingeridos';")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Capa de datos limpios y tipados';")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Capa de agregaciones de negocio';")

print("Catálogos y esquemas organizados con éxito.")

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
# GRANT 1
spark.sql("GRANT SELECT ON SCHEMA wanderbricks_catalog.gold TO `alvaromunoz0405@gmail.com`;")

# GRANT 2
spark.sql("GRANT ALL PRIVILEGES ON SCHEMA wanderbricks_catalog.bronze TO `alvaromunoz0405@gmail.com`;")

print("--- Permisos en la capa Oro ---")
display(spark.sql("SHOW GRANTS ON SCHEMA wanderbricks_catalog.gold"))

print("--- Permisos en la capa Bronce ---")
display(spark.sql("SHOW GRANTS ON SCHEMA wanderbricks_catalog.bronze"))

### 4.3 Linaje

*Evidenciar el linaje de una tabla desde el Catalog Explorer. Insertar la captura.*
![Linaje 1](https://i.postimg.cc/kV36M48T/IMAGEN-3.png)
![Linaje 2](https://i.postimg.cc/bDqZbBmJ/IMAGEN-1.png)

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida. Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*
![Automatizacion](https://i.postimg.cc/k6728pTD/IMAGEN-2.png)

In [0]:
# Crear tablas reales para que Databricks dibuje el linaje
spark.sql("CREATE TABLE IF NOT EXISTS wanderbricks_catalog.bronze.raw_ventas (id INT, monto DOUBLE)")
spark.sql("INSERT INTO wanderbricks_catalog.bronze.raw_ventas VALUES (1, 100)")
spark.sql("CREATE TABLE IF NOT EXISTS wanderbricks_catalog.silver.clean_ventas AS SELECT * FROM wanderbricks_catalog.bronze.raw_ventas")

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

In [0]:
# Código simulado de las tareas que ejecuta el Job
print("Ejecutando Tarea 1 (Bronze -> Silver): Limpieza de datos...")
# spark.sql("INSERT INTO wanderbricks_catalog.silver.clean_data SELECT ...")

print("Ejecutando Tarea 2 (Silver -> Gold): Agregaciones y KPIs...")
# spark.sql("INSERT INTO wanderbricks_catalog.gold.kpis SELECT ...")

print("Job finalizado exitosamente.")

---
## 5. Resultados
Se implementó el entorno completo configurando la arquitectura en tres capas dentro de Unity Catalog. Los comandos `GRANT` se aplicaron limitando el acceso según el rol. El linaje visual documentó la trazabilidad, y la automatización mediante Jobs permitió encadenar el flujo de datos sin intervención manual.

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*
1. **Gobierno centralizado:** Unity Catalog facilita una gestión de accesos mucho más eficiente y segura que la administración manual en servidores IaaS.
2. **Eficiencia en PaaS:** Desplegar los recursos tomó minutos frente a las horas requeridas para montar y afinar un clúster de Spark en máquinas virtuales.
3. **Trazabilidad:** El linaje de datos generado automáticamente mejora la auditoría y asegura la confianza en las métricas de la capa Gold.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Alvaro Enrique Muñoz Giraldo | Configuración Unity Catalog, Permisos y Job | Explica la arquitectura administrada y muestra el entorno en vivo |
| [Nombre 2] | Diseño de esquemas Medallion y linaje | Explica el GRANT ejecutado y el linaje de datos |
| [Nombre 3] | Comparativa técnica y documentación | Explica el equivalente IaaS y las complicaciones de VM |

**Uso de asistentes de IA:** Se utilizó un asistente de IA para revisar la sintaxis de SQL de Unity Catalog y estructurar los puntos de la comparativa técnica entre IaaS y PaaS.

---
## 📹 Preguntas obligatorias de sustentación

**1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?**
El proveedor administra los servidores físicos, el almacenamiento subyacente y la elasticidad de los clusters. Nosotros administramos el modelado de datos, los permisos en Unity Catalog y la lógica de los pipelines.

**2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.**
Ejecuté `GRANT SELECT ON SCHEMA wanderbricks_catalog.gold TO account_users;`. Le da acceso de solo lectura a los analistas sobre la capa Oro, asegurando que consulten las métricas sin modificar los datos crudos.

**3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?**
La configuración y sincronización del clúster distribuido de Apache Spark, mantener la red segura mediante firewalls manuales, aplicar parches de sistema operativo y manejar la falta de auto-escalado ante cargas variables.

---
## ✅ Antes de entregar

- [x] El diagrama de arquitectura está incluido y descrito
- [x] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [x] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [x] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [x] El enlace del video está en la portada y abre desde otra cuenta
- [x] Todo confirmado en /ea2 y el HTML subido a Canvas